In [5]:
import os
import random
import warnings

import torch
import torch.nn.functional as F

from tqdm import tqdm
from transformers import AutoModel, AutoProcessor
from torchvision import datasets
from torchvision.io import decode_image

from torch.export import export
from torchao.quantization.pt2e.quantize_pt2e import convert_pt2e, prepare_pt2e

from executorch import version
from executorch.runtime import Runtime
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import XNNPACKQuantizer, get_symmetric_quantization_config


random.seed(0)
warnings.simplefilter("ignore", FutureWarning)
print("Executorch version: ", version.__version__)

Executorch version:  1.1.0


In [6]:
SUFFIX = "_xnnpack_int8"
EXPORT_DIR = "exports"
MODEL_PATH: dict[str, str] = {}

processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")


class ClipVisionModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        model = AutoModel.from_pretrained("openai/clip-vit-base-patch32")
        self.vision_model = model.vision_model
        self.visual_projection = model.visual_projection

    def forward(self, pixel_values):
        model_output = self.vision_model(pixel_values)
        pooler_output = model_output.pooler_output
        projected_output = self.visual_projection(pooler_output)
        return F.normalize(projected_output, p=2, dim=1)

In [7]:
print("Exporting model...")

# We use a common dataset of images for calibration of activation ranges.
#
# NOTE: There is some problem with getting the VOCSegmentation dataset to
# download in torchvision (https://github.com/pytorch/vision/issues/9269). You
# can download it from a mirror using the following commands:
# ```
# mkdir -p datasets/VOCSegmentation
# cd datasets/VOCSegmentation
# curl https://dataset.bj.bcebos.com/voc/VOCtrainval_11-May-2012.tar
# ```
#
calibration_dataset = datasets.VOCSegmentation(root="datasets/VOCSegmentation", year="2012", image_set="train")
calibration_size = 1_000
calibration_imgs = random.sample(calibration_dataset.images, k=calibration_size)

model = ClipVisionModel().eval()
inputs = (processor(images=torch.randint(0, 256, (1, 3, 224, 224)), return_tensors="pt")["pixel_values"],)

# NOTE: Quantization pipeline is based on the ExecuTorch example for xnnpack
# backend:
# https://docs.pytorch.org/executorch/1.1/backends/xnnpack/xnnpack-quantization.html

# Prepare quantizer
# -----------------
# NOTE: We quantize only nn.Linear modules rather than using set_global. Global
# quantization inserts quantize/dequantize ops around every op including the
# attention reshape/transpose ops, causing xnn_status_invalid_parameter in the
# XNNPACK compiler.
quantizer = XNNPACKQuantizer()
quantizer.set_module_type(torch.nn.Linear, get_symmetric_quantization_config(is_per_channel=True))

# Prepare model for PTQ
# ---------------------
training_ep = export(model, inputs).module()
prepared_model = prepare_pt2e(training_ep, quantizer)

# Calibrate activation ranges
# ---------------------------
for img_path in tqdm(calibration_imgs, desc="Calibrating"):
    img = decode_image(img_path)
    prepared_model(processor(images=img, return_tensors="pt")["pixel_values"])

# Export quantized model
# ----------------------
quantized_model = convert_pt2e(prepared_model)

program = to_edge_transform_and_lower(export(quantized_model, inputs), partitioner=[XnnpackPartitioner()])
program = program.to_executorch()

os.makedirs(EXPORT_DIR, exist_ok=True)
with open(path := f"{EXPORT_DIR}/clip_vit_base_patch32_vision{SUFFIX}.pte", "wb") as f:
    MODEL_PATH["clip_vision"] = path
    f.write(program.buffer)

print(f"Exported {path}")

Exporting model...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 64570.18it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Calibrating: 100%|██████████| 1000/1000 [01:34<00:00, 10.54it/s]


Exported exports/clip_vit_base_patch32_vision_xnnpack_int8.pte


In [8]:
import time

print("Benchmarking...")

path = MODEL_PATH["clip_vision"]
runtime = Runtime.get()
et_program = runtime.load_program(path)
method = et_program.load_method("forward")

pixel_values = processor(images=torch.randint(0, 256, (1, 3, 224, 224)), return_tensors="pt")["pixel_values"]

tik = time.perf_counter()
for _ in range(100):
    method.execute([pixel_values])  # type: ignore
tok = time.perf_counter()

print(
    f"{path}\n"
    f"Inference time (avg over 100 runs): {(tok - tik) / 100 * 1000:.2f} ms\n"
    f"Model size: {os.path.getsize(path) / 1024 / 1024:.2f} MB\n"
)

[program.cpp:154] InternalConsistency verification requested but not available
[cpuinfo_utils.cpp:71] Reading file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:87] Failed to open midr file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:167] Number of efficient cores 4


Benchmarking...
exports/clip_vit_base_patch32_vision_xnnpack_int8.pte
Inference time (avg over 100 runs): 23.67 ms
Model size: 91.89 MB



In [10]:
print("Evaluating quality...")

# Measure cosine similarity between fp32 and int8 embeddings on VOC val images.
# A score close to 1.0 indicates minimal degradation from quantization.
evaluation_dataset = datasets.VOCSegmentation(root="datasets/VOCSegmentation", year="2012", image_set="val")
evaluation_size = 100
evaluation_imgs = random.sample(evaluation_dataset.images, k=evaluation_size)

path = MODEL_PATH["clip_vision"]
runtime = Runtime.get()
et_program = runtime.load_program(path)
method = et_program.load_method("forward")

cos_sims = []
for img_path in tqdm(evaluation_imgs, desc="Evaluating"):
    img = decode_image(img_path)
    pixel_values = processor(images=img, return_tensors="pt")["pixel_values"]

    with torch.no_grad():
        fp32_emb = model(pixel_values)

    int8_emb, *_ = method.execute([pixel_values])  # type: ignore
    cos_sims.append(F.cosine_similarity(fp32_emb, int8_emb).item())

mean_cos_sim = sum(cos_sims) / len(cos_sims)
print(
    "----------------------------------\n"
    f"Mean cosine similarity (fp32 vs int8): {mean_cos_sim:.4f}\n"
    f"Min cosine similarity:                 {min(cos_sims):.4f}\n"
    f"Max cosine similarity:                 {max(cos_sims):.4f}\n"
    f"Model size:                            {os.path.getsize(path) / 1024 / 1024:.2f} MB\n"
    "----------------------------------"
)

[program.cpp:154] InternalConsistency verification requested but not available


Evaluating quality...


Evaluating: 100%|██████████| 100/100 [00:05<00:00, 17.05it/s]

----------------------------------
Mean cosine similarity (fp32 vs int8): 0.9427
Min cosine similarity:                 0.8354
Max cosine similarity:                 0.9804
Model size:                            91.89 MB
----------------------------------
